<a href="https://colab.research.google.com/github/arups330/ElitLab_MED_VQA/blob/main/BrainFace(closed)_WITHOUT_CoT_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Colab] Closed-question fine-tuning **WITHOUT CoT** — all models, one after another, auto-push to Hugging Face

**Before running**
* Runtime → Change runtime type → **GPU** (T4).
* Upload your dataset **.zip** to `/content` via the Files panel (left sidebar). Cell 5 extracts it and finds `closed_with_CoT.csv` automatically.
* 🔑 **Secrets** (left sidebar) → add `HF_TOKEN` = your Hugging Face **Write** token → enable *Notebook access*.
* Accept the license at https://huggingface.co/google/medgemma-4b-it.

Cells 7–14 are each one step of the original notebook (load → LoRA → before check → trainer → train → after check → save → push → free GPU). Cell 15 runs them for every model in `REPO_IDS`, pushing each to the Hub before the next. Models already on the Hub are skipped — if the summary shows an OOM, **Runtime → Restart session** and run again.

In [1]:
!pip install -q "unsloth==2026.9.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # before torch is imported
os.environ["UNSLOTH_RETURN_LOGITS"]   = "1"   # bypass Unsloth fused CE loss (Llama torch.compile crash)
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"   # disable torch.compile patches (same maths, a bit slower)

from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch, gc, shutil

REPO_IDS = {
    "MedGemma-4B": "google/medgemma-4b-it",
    "Qwen2.5-VL-7B": "Qwen/Qwen2.5-VL-7B-Instruct",
    "Llama-11B": "unsloth/Llama-3.2-11B-Vision-Instruct",
    "Qwen3-VL-8B": "Qwen/Qwen3-VL-8B-Instruct",
    "Lingshu-7B": "lingshu-medical-mllm/Lingshu-7B",
    # "Gemma4-E4B": "google/gemma-4-E4B-it",   # not trainable on T4 -> needs Colab L4/A100
}

HF_USERNAME = "Arup330"
DATASET_TAG = "BrainFace"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# HF token from Colab Secrets (🔑 icon in left sidebar -> add HF_TOKEN, enable notebook access)
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

In [4]:
# ---------------------------------------------------------------------------
# 3. Upload zip -> extract -> find closed_with_CoT.csv -> keep Yes/No rows only
# ---------------------------------------------------------------------------
import os
import glob
import zipfile
import pandas as pd
from PIL import Image

EXTRACT_DIR = "/content/data"

# Use a zip already uploaded to /content (Files panel), otherwise open the upload dialog
zips = glob.glob("/content/*.zip")
if not zips:
    from google.colab import files
    uploaded = files.upload()
    zips = [f"/content/{n}" for n in uploaded.keys()]
ZIP_PATH = zips[0]
print("Using zip:", ZIP_PATH)

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)
print("Extracted to", EXTRACT_DIR)

# Locate closed.csv anywhere inside the extracted folder (case-insensitive)
csv_paths = [p for p in glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True)
             if os.path.basename(p).lower() == "closed.csv"]
assert csv_paths, "closed.csv not found inside the zip"
print(f"Found {len(csv_paths)} CSVs:")
for p in csv_paths:
    print(" ", p)

frames = []
for csv_path in csv_paths:
    split_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    df["split_dir"] = split_dir
    frames.append(df)

cot_df = pd.concat(frames, ignore_index=True)

# Keep ONLY closed questions whose answer is Yes / No -- ignore everything else
before = len(cot_df)
ans = cot_df["answer"].astype(str).str.strip().str.lower()
cot_df = cot_df[ans.isin(["yes", "no"])].copy()
cot_df["answer"] = cot_df["answer"].astype(str).str.strip().str.lower().map({"yes": "Yes", "no": "No"})
cot_df = cot_df.reset_index(drop=True)
print(f"Kept {len(cot_df)} / {before} Yes/No rows "
      f"(Yes={(cot_df['answer']=='Yes').sum()}, No={(cot_df['answer']=='No').sum()})")

IMG_COL = "image_file" if "image_file" in cot_df.columns else "img_name"

# Index every image inside the zip by file name (images may be in any sub-folder)
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".gif", ".webp")
IMG_INDEX = {os.path.basename(p): p
             for p in glob.glob(os.path.join(EXTRACT_DIR, "**", "*"), recursive=True)
             if p.lower().endswith(IMG_EXTS)}
print(f"Indexed {len(IMG_INDEX)} image files")

def resolve_image_path(split_dir: str, img_name: str) -> str:
    flat_name = os.path.basename(str(img_name))
    candidates = [
        os.path.join(split_dir, str(img_name)),
        os.path.join(split_dir, flat_name),
        os.path.join(split_dir, str(img_name).replace("/", "_")),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    if flat_name in IMG_INDEX:
        return IMG_INDEX[flat_name]
    raise FileNotFoundError(f"Could not find image for {img_name!r} in {split_dir}")

print("First image:", resolve_image_path(cot_df.iloc[0]["split_dir"], cot_df.iloc[0][IMG_COL]))

Using zip: /content/BrainFaceT_train_with_CoT.zip
Extracted to /content/data
Found 1 CSVs:
  /content/data/BrainFaceT_train_with_CoT/CT/train/closed.csv
Kept 62 / 64 Yes/No rows (Yes=22, No=40)
Indexed 11 image files
First image: /content/data/BrainFaceT_train_with_CoT/CT/train/xmlab403_source.jpg


In [5]:
# ---------------------------------------------------------------------------
# 4. convert_to_conversation -- same style as Unsloth's radiography example,
#    but: instruction = your closed-question prompt, and the
#    assistant target is the one-word ground-truth answer (Yes / No).
# ---------------------------------------------------------------------------
def systemPrompt(question: str) -> str:
    return f"""
Context:
- You are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.

Objective:
- Answer the user's question by verifying whether it is supported by the visual evidence in the medical image.

Inputs:
- Question: {question}

You have to think step by step.
Instructions:
1. Examine the medical image carefully.
2. Independently determine the relevant visual findings before considering the CoT.
3. Compare your own observations with the provided CoT.
4. If the CoT is inconsistent with the image, disregard it.
5. Answer the question using the following evidence priority:
1. Medical image (highest priority)
2. User question
3. CoT (only if verified by the image)
6. Never fabricate findings or rely on assumptions.
7. If the image does not provide sufficient evidence to support a "Yes" answer, return "No."

Output Requirements:
- Output exactly one word.
- Do not provide explanations, punctuation, or additional text.

Valid outputs:
Yes
No

"""

def convert_to_conversation(sample):
    instruction = systemPrompt(sample["question"])
    image_path = resolve_image_path(sample["split_dir"], sample[IMG_COL])
    image = Image.open(image_path).convert("RGB")

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": image},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["answer"]}],
        },
    ]
    return {"messages": conversation}
pass

print("Converting rows to Unsloth chat format...")
converted_dataset = [convert_to_conversation(row) for _, row in cot_df.iterrows()]
print(f"Done. {len(converted_dataset)} examples ready.")
print("Example:", converted_dataset[0]["messages"])

Converting rows to Unsloth chat format...
Done. 62 examples ready.
Example: [{'role': 'user', 'content': [{'type': 'text', 'text': '\nContext:\n- You are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.\n\nObjective:\n- Answer the user\'s question by verifying whether it is supported by the visual evidence in the medical image.\n\nInputs:\n- Question: Is this an MRI?\n\nYou have to think step by step.\nInstructions:\n1. Examine the medical image carefully.\n2. Independently determine the relevant visual findings before considering the CoT.\n3. Compare your own observations with the provided CoT.\n4. If the CoT is inconsistent with the image, disregard it.\n5. Answer the question using the following evidence priority:\n1. Medical image (highest priority)\n2. User question\n3. CoT (only if verified by the image)\n6. Never fabricate findings or rely on assumptions.\n7. If 

## Per-model steps (each cell = one step of the original notebook)

In [6]:
# ---------------------------------------------------------------------------
# 1. Load model
# ---------------------------------------------------------------------------
def load_model(repo_id):
    extra = {}
    if "Qwen3-VL" in repo_id:
        extra["device_map"] = {"": 0}   # Qwen3-VL vision encoder breaks when split across GPUs
    model, tokenizer = FastVisionModel.from_pretrained(
        repo_id,
        load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
        token = HF_TOKEN,
        **extra,
    )
    return model, tokenizer

In [7]:
# ---------------------------------------------------------------------------
# 2. Attach LoRA adapters
# ---------------------------------------------------------------------------
def attach_lora(model, model_name):
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=(model_name != "MedGemma-4B"),   # T4 has no bf16: keep MedGemma's vision tower frozen
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )
    if model_name == "MedGemma-4B":
        # T4 trains Gemma3 in float32; upcast any bf16 weights left in the vision tower so layer_norm dtypes match
        for p in model.parameters():
            if p.dtype == torch.bfloat16:
                p.data = p.data.float()
        for b in model.buffers():
            if b.dtype == torch.bfloat16:
                b.data = b.data.float()
    return model

In [8]:
# ---------------------------------------------------------------------------
# 5. Quick check BEFORE fine-tuning (same as the Unsloth notebook does --
#    run one inference with the base model to see what it currently outputs)
# ---------------------------------------------------------------------------
def check_before(model, tokenizer):
    FastVisionModel.for_inference(model)
    sample = cot_df.iloc[0]
    test_instruction = systemPrompt(sample["question"])
    test_image = Image.open(resolve_image_path(sample["split_dir"], sample[IMG_COL])).convert("RGB")

    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": test_instruction}
    ]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(test_image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    output_ids = model.generate(**inputs, max_new_tokens=256, use_cache=True, temperature=1.5, min_p=0.1)
    print("\nBEFORE fine-tuning, model output:")
    print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    print("Ground truth:", sample["answer"])
    return inputs, sample

In [9]:
# ---------------------------------------------------------------------------
# 6. Train with SFTTrainer + UnslothVisionDataCollator
# ---------------------------------------------------------------------------
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator

def build_trainer(model, tokenizer):
    FastVisionModel.for_training(model)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer),
        train_dataset=converted_dataset,
        args=SFTConfig(
            per_device_train_batch_size=2,   # CoT targets are long -- keep batch size low
            gradient_accumulation_steps=4,
            warmup_steps=5,
            #max_steps=30,                  # quick test run -- comment out for full training
            num_train_epochs=1,
            learning_rate=2e-4,
            fp16=not is_bf16_supported(),
            bf16=is_bf16_supported(),
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir="outputs",
            report_to="none",

            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            dataset_num_proc=4,
            max_length=4096,   # CoT outputs are long (9 sections) -- increased from 2048 (TRL name: max_length)
        ),
    )
    return trainer

In [10]:
# ---------------------------------------------------------------------------
# 7. Check AFTER fine-tuning -- same sample as before, compare outputs
# ---------------------------------------------------------------------------
def check_after(model, tokenizer, inputs, sample):
    FastVisionModel.for_inference(model)
    output_ids = model.generate(**inputs, max_new_tokens=1024, use_cache=True, temperature=0.3, min_p=0.1)
    print("\nAFTER fine-tuning, model output:")
    print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    print("\n(Ground truth answer it was trained on:)")
    print(sample["answer"])

In [11]:
# ---------------------------------------------------------------------------
# 8. Save the LoRA adapter locally
# ---------------------------------------------------------------------------
def save_local(model, tokenizer, local_dir):
    model.save_pretrained(local_dir)
    tokenizer.save_pretrained(local_dir)
    print(f"\nSaved locally to ./{local_dir}")

In [12]:
# ---------------------------------------------------------------------------
# 9. Push to Hugging Face Hub
# ---------------------------------------------------------------------------
def push_to_hf(model, tokenizer, hub_repo):
    model.push_to_hub(hub_repo, token=HF_TOKEN)
    tokenizer.push_to_hub(hub_repo, token=HF_TOKEN)
    print(f"Pushed -> https://huggingface.co/{hub_repo}")

In [13]:
# ---------------------------------------------------------------------------
# 10. Free GPU memory + HF download cache before the next model
# ---------------------------------------------------------------------------
def free_gpu():
    for v in ["trainer", "model", "tokenizer", "inputs"]:
        if v in globals(): del globals()[v]
    gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)   # keeps Colab disk free
    print("GPU free after cleanup:", [f"{torch.cuda.mem_get_info(i)[0]/1e9:.1f} GB" for i in range(torch.cuda.device_count())])

## Run all models (one after another; each pushed to the Hub before the next starts)

In [14]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
results = {}

for model_name, repo_id in REPO_IDS.items():
    HUB_REPO  = f"{HF_USERNAME}/{DATASET_TAG}_closed_noCoT_{model_name}_lora"
    LOCAL_DIR = f"{DATASET_TAG}_closed_noCoT_{model_name}_lora"

    if api.repo_exists(HUB_REPO):          # already trained + pushed -> skip (safe re-run after a session ends)
        print(f"\n[SKIP] {model_name} already on Hub: https://huggingface.co/{HUB_REPO}")
        results[model_name] = "skipped"
        continue

    print("\n" + "="*70)
    print(f"MODEL: {model_name}  ({repo_id})  ->  {HUB_REPO}")
    print("="*70)

    try:
        model, tokenizer = load_model(repo_id)                 # 1
        model            = attach_lora(model, model_name)      # 2
        inputs, sample   = check_before(model, tokenizer)      # 5
        trainer          = build_trainer(model, tokenizer)     # 6
        trainer_stats    = trainer.train()
        check_after(model, tokenizer, inputs, sample)          # 7
        save_local(model, tokenizer, LOCAL_DIR)                # 8
        push_to_hf(model, tokenizer, HUB_REPO)                 # 9
        results[model_name] = "OK"
    except Exception as e:
        import traceback; traceback.print_exc()
        results[model_name] = f"FAILED: {e}"

    free_gpu()                                                 # 10

print("\n================ SUMMARY ================")
for k, v in results.items():
    print(f"{k:15s} {v}")


MODEL: MedGemma-4B  (google/medgemma-4b-it)  ->  Arup330/BrainFace_closed_noCoT_MedGemma-4B_lora
==((====))==  Unsloth 2026.9.2: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]


BEFORE fine-tuning, model output:
The image is a CT scan of the head.

No

Ground truth: No


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 62 | Num Epochs = 1 | Total steps = 8
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,802,496 of 4,329,881,968 (0.69% trained)
Unsloth: Not an error, but Gemma3ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


...

Unsloth: Offloading embeddings to RAM to save 1.02 GB.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.



BEFORE fine-tuning, model output:
No.
Ground truth: No
Unsloth: Model does not have a default image size - using 512


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 62 | Num Epochs = 1 | Total steps = 8
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 51,521,536 of 8,343,688,192 (0.62% trained)
Traceback (most recent call last):
  File "/tmp/ipykernel_3418/627231972.py", line 23, in <cell line: 0>
    trainer_stats    = trainer.train()
  File "/usr/local/lib/python3.13/dist-packages/unsloth/trainer.py", line 1049, in _train_with_reset
    return _orig_train(*train_args, **train_kwargs)
  File "/usr/local/lib/python3.13/dist-packages/transformers/trainer.py", line 1424, in train
    return inner_training_loop(
        args=args,
    ...<2 lines>...
        ignore_keys_for_eval=ignore_keys_for_eval,
    )
  File "<string>", line 81, in _fast_inner_training_loop
  File "/usr/local/lib/python3.13/dist-packages/transformers

GPU free after cleanup: ['5.8 GB']

================ SUMMARY ================
MedGemma-4B     OK
Qwen2.5-VL-7B   OK
Llama-11B       FAILED: CUDA out of memory. Tried to allocate 252.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 231.81 MiB is free. Including non-PyTorch memory, this process has 14.33 GiB memory in use. Of the allocated memory 14.06 GiB is allocated by PyTorch, and 113.04 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
Qwen3-VL-8B     OK
Lingshu-7B      FAILED: CUDA out of memory. Tried to allocate 678.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 411.81 MiB is free. Including non-PyTorch memory, this process has 14.16 GiB memory in use. Of the allocated memory 13.80 GiB is alloc